In [12]:
import pandas as pd
import os
import numpy as np

for LDSC heritability, we need to provide an annot file to estimate heritability from specific categories

## Get significant snps

In [13]:
mp_ldsc={"eur":"EUR","afr":"AFR","his":"AMR"}

sumstats_dir="/cellar/users/mpagadal/projects/TestosteroneGWAS/discovery/data/summarystats/full/"
ldsc_dir="/cellar/users/mpagadal/data/ldsc/1000G_plink/"

In [14]:
for group in mp_ldsc.keys():
    print(group)
    ref_group=mp_ldsc[group]
    direct=ldsc_dir+ref_group+"/"
    
    #get summary statistics
    summary=pd.read_csv(sumstats_dir+"compiled."+group+".rsid.release4.testosterone.glm.linear",delimiter="\t")
    summary["#CHROM"]=summary["#CHROM"].replace("X",23)
    summary["ID"]=summary["#CHROM"].astype(str)+":"+summary["POS"].astype(str)+":"+summary["REF"]+":"+summary["ALT"]
    
    #get significant snps
    sig_thresh=.00000005
    sig_snps=summary[summary["P"]<sig_thresh]["ID"].tolist()+[x.rsplit(":",2)[0]+":"+x.split(":")[3]+":"+x.split(":")[2] for x in summary[summary["P"]<sig_thresh]["ID"].tolist()]
    
    #get significant snps
    sug_thresh=.00001
    sug_snps=summary[summary["P"]<sug_thresh]["ID"].tolist()+[x.rsplit(":",2)[0]+":"+x.split(":")[3]+":"+x.split(":")[2] for x in summary[summary["P"]<sug_thresh]["ID"].tolist()]
    
    #iterate through bim files and get snp sets
    files=[x for x in os.listdir(direct) if "bim" in x]
    files=[x for x in files if "CM" in x]
    print(files)
    
    for x in files:
        print(x)
        #reformat bim
        df=pd.read_csv(direct+x,delimiter="\t",header=None)
        df["snp"]=df[0].astype(str)+":"+df[3].astype(str)+":"+df[4]+":"+df[5]
    
        #make annot file
        df["base"]=1
        df["significant"]=np.where(df["snp"].isin(sig_snps),1,0)
        df["suggestive"]=np.where(df["snp"].isin(sug_snps),1,0)
    
        annot=df[[0,3,1,2,"base","suggestive","significant"]]
        annot.columns=["CHR","BP","SNP","CM","base","suggestive","significant"]
        print(annot["significant"].value_counts())
        print(annot["suggestive"].value_counts())
        annot.to_csv(direct+"baseline."+str(x.split(".")[2])+".annot",index=None,sep="\t")
    

eur
['1000G.EUR.5.CM.bim', '1000G.EUR.20.CM.bim', '1000G.EUR.9.CM.bim', '1000G.EUR.11.CM.bim', '1000G.EUR.10.CM.bim', '1000G.EUR.2.CM.bim', '1000G.EUR.3.CM.bim', '1000G.EUR.19.CM.bim', '1000G.EUR.8.CM.bim', '1000G.EUR.15.CM.bim', '1000G.EUR.14.CM.bim', '1000G.EUR.17.CM.bim', '1000G.EUR.4.CM.bim', '1000G.EUR.7.CM.bim', '1000G.EUR.13.CM.bim', '1000G.EUR.22.CM.bim', '1000G.EUR.16.CM.bim', '1000G.EUR.6.CM.bim', '1000G.EUR.1.CM.bim', '1000G.EUR.X.CM.bim', '1000G.EUR.21.CM.bim', '1000G.EUR.18.CM.bim', '1000G.EUR.12.CM.bim']
1000G.EUR.5.CM.bim
0    5238706
Name: significant, dtype: int64
0    5238654
1         52
Name: suggestive, dtype: int64
1000G.EUR.20.CM.bim
0    1803869
Name: significant, dtype: int64
0    1803862
1          7
Name: suggestive, dtype: int64
1000G.EUR.9.CM.bim
0    3542185
Name: significant, dtype: int64
0    3542111
1         74
Name: suggestive, dtype: int64
1000G.EUR.11.CM.bim
0    4024959
Name: significant, dtype: int64
0    4024842
1        117
Name: suggestive, dty

### rename frq files

In [17]:
frq_files=[x for x in os.listdir("/cellar/users/mpagadal/data/ldsc/1000G_plink/AMR_frq/") if "frq.frq" in x]

In [18]:
frq_files

['1000G.AMR.17.frq.frq.gz',
 '1000G.AMR.1.frq.frq.gz',
 '1000G.AMR.16.frq.frq.gz',
 '1000G.AMR.7.frq.frq.gz',
 '1000G.AMR.X.frq.frq.gz',
 '1000G.AMR.3.frq.frq.gz',
 '1000G.AMR.20.frq.frq.gz',
 '1000G.AMR.2.frq.frq.gz',
 '1000G.AMR.9.frq.frq.gz',
 '1000G.AMR.14.frq.frq.gz',
 '1000G.AMR.8.frq.frq.gz',
 '1000G.AMR.22.frq.frq.gz',
 '1000G.AMR.15.frq.frq.gz',
 '1000G.AMR.11.frq.frq.gz',
 '1000G.AMR.18.frq.frq.gz',
 '1000G.AMR.21.frq.frq.gz',
 '1000G.AMR.4.frq.frq.gz',
 '1000G.AMR.19.frq.frq.gz',
 '1000G.AMR.6.frq.frq.gz',
 '1000G.AMR.10.frq.frq.gz',
 '1000G.AMR.13.frq.frq.gz',
 '1000G.AMR.5.frq.frq.gz',
 '1000G.AMR.12.frq.frq.gz']

In [21]:
for x in frq_files:
    os.rename("/cellar/users/mpagadal/data/ldsc/1000G_plink/AMR_frq/"+x,"/cellar/users/mpagadal/data/ldsc/1000G_plink/AMR_frq/"+x.split(".frq")[0]+".frq.gz")
    print(x.split(".frq")[0])

1000G.AMR.17
1000G.AMR.1
1000G.AMR.16
1000G.AMR.7
1000G.AMR.X
1000G.AMR.3
1000G.AMR.20
1000G.AMR.2
1000G.AMR.9
1000G.AMR.14
1000G.AMR.8
1000G.AMR.22
1000G.AMR.15
1000G.AMR.11
1000G.AMR.18
1000G.AMR.21
1000G.AMR.4
1000G.AMR.19
1000G.AMR.6
1000G.AMR.10
1000G.AMR.13
1000G.AMR.5
1000G.AMR.12
